# Efficiency Simulation
_Author: Danaé Valdenaire, Philipp Schreiner_<br>
_Created: Sep. 5, 2025_<br>
_Last updated: June. 15th, 2026 by Danaé Valdenaire_

---

## Introduction

This tutorial walks you through the **efficiency simulation** using the CAIT software.
But first, a little bit of context is required.

Not all particle events end up in the final spectrum. Some fall below the **trigger
threshold**, others are removed by **quality cuts**. To correctly interpret the measured
spectrum — for example when computing dark matter exclusion limits — the probability
of an event surviving this analysis chain must be estimated as a function of energy.
This quantity is called the **survival probability**, or **efficiency**.

The efficiency is estimated by simulation: artificial pulses of known energy are
superimposed onto the real data stream at random timestamps, then processed through
the **same analysis chain** as real data. Since their true energies are known, the
surviving fraction can be extracted as a function of energy.

```{tip}
For more the algorithm, go to [**`dh.efficiency_sim_trigger_of`**](cait.mixins.SimulateMixin.efficiency_sim_trigger_of)
```

## First, you need data

This part is from the triggering stream data tutorial.

In [ ]:
# One hour of stream data with two channels:
# Here, we set a random seed so that the tutorial looks the same for you.
# If you want to trigger actual data, you want to construct either of the
# stream objects explained above, depending on the hardware used.
stream = vai.MockStream(seed=137, rate_Hz=2)

# You can check which channels are present in the stream ...
print(f"Available channels: {stream.keys}")
# ... which testpulse channels are available ...
print(f"Available TP channels: {stream.tp_keys}")
# ... and which TPAs are available:
print(f"Available TPAs:", {k: np.unique(v) for k, v in stream.tpas.items()})

# Just putting the stream object at the end of a cell gives you 
# an information overview. E.g. the timebase 'dt_us' and the 
# measuring time in hours 'measuring_time_h'.
stream

In [ ]:
# Configure trigger
record_length = 2**14

# Basic configuration
trigger_config = {
    "trigger_channels": ["Ch0"], # those will be triggered, e.g. phonon channel
    "passive_channels": ["Ch1"], # those will be read in coincidence, e.g. light channel
    "testpulse_channels": ["TP0", "TP1"], # the testpulse channels corresponding to all trigger/passive channels
    "controlpulses_above": [9., 9.], # controlpulses have TPA=10 (see above)
    "f_noise": 1000, # we will sample 1000 random noise traces per hour (later needed for NPS creation)
    "copy_events": True, # set this to False if you don't want to copy the raw data of the stream to the HDF5 file (to save disk space)
}

In [ ]:
# Path to where we want to save the HDF5 file
fdirh5 = "tutorial_output"
hdf5_name = "my_first_trigger"

os.makedirs(fdirh5, exist_ok=True)

dh = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=len(trigger_config["trigger_channels"]) + len(trigger_config["passive_channels"]), 
                    sample_frequency=stream.sample_frequency)
dh.set_filepath(path_h5=fdirh5, 
                fname=hdf5_name, 
                appendix=False)
dh.init_empty()
print(dh)

In [ ]:
dh.trigger_zscore(stream, **trigger_config)

In [ ]:
for group in ["events", "testpulses", "noise", "controlpulses"]:
    dh.cmp(group)

## Simulating timestamps and pulse heights

In the previous notebook ```new_tutorial_energy_calibration```, you learned how to perform the energy calibration of your data and save your ```e_cal``` function as a ```.json```file. If you did not completed this part, please run the ```new_tutorial_energy_calibration``` notebook first and come back here afterwards.

Let's start by loading our calibration function that enable us to convert from pulse heights to testpulse equivalent and vice versa.

In [ ]:
my_ecal = vai.EnergyCalibration.from_file("my_ecal_function")

We need to generate random (sorted) timestamps. The simulation is performed on a chunk of length ```n_record_lens``` and placed according to ```record_placement```. 

```{important}
The following definition of generated timestamps is assuming default values of ```n_record_lens``` and ```record_placement```. 
```

You also need to choose the number of event you want to simulate. To get enough statistics, this number needs to be high. 

In [ ]:
N_sim = 10**6

In [ ]:
sim_ts = np.sort(
    sp.stats.randint.rvs(
        stream.time[0] + 6*stream.dt_us*record_length, 
        stream.time[-1] - 4*stream.dt_us*record_length, 
        size=N_sim
    )
)

The efficiency simulation can be perform in volt or in energy. In this case, we will mimic a real analysis and therefore work in energy unit. 
- First, we define the range of energy (in keV) in which we want to run the simulation. We have to stay in the linear range of the detector otherwise the pulse shape of the particle events and the simulated pulses will differ. 
- Then, we convert this energy array to testpulse equivalent using our ```e_cal``` function. The ```e_cal``` function will incorporate the instabilities of the detector over time. We start with a uniform distribution of energies but we will get a non-uniform TPE distribution. 
- The last step is to convert TPEs to pulse heights, by dividing with the CPE factor. This value is the ratio between the energy of the peak (in keV) and the position of the peak (in V) in the TPE spectrum. You should have it from the energy calibration tutorial.

In [2]:
cpe_factor = 6.5/2 #[keV/V]

In [ ]:
energy_range = (0, 2) # Energy range to perform the simulation (keV)
energy_of_interest = energy_range[0] + np.diff(energy_range)*sp.stats.uniform.rvs(size=N_sim)
tpes_of_interest = energy_of_interest/cpe_factor
phs_of_interest = my_ecal(timestamps_of_interest, tpes_of_interest)

## Trigger efficiency

... explain how to run [**`dh.efficiency_sim_trigger_of`**](cait.mixins.SimulateMixin.efficiency_sim_trigger_of), how to use its `preview` argument to judge the correctness of the simulation, which groups are generated in the [**`DataHandler`**](cait.DataHandler), what the datasets mean, and how they can be used to extract information.

In [ ]:
dh.efficiency_sim_trigger_of(
                stream=stream,
                trigger_channels=["Ch0"],
                passive_channels=["Ch1"],
                testpulse_channels=["TP0", "TP1"],
                of=of,
                thresholds=[0.1],
                sim_ts=sim_ts,
                sim_phs=sim_phs,
                #preview=True, # Uncomment to see a preview of the trigger before you run the simulation
                sev_fitpars=sev_fitpars,
                shift_samples=shifts,
            )

Two groups have been created in the DataHandler ```trig-eff-sim``` contains trigger information and the simulation chunks, ```events-eff-sim``` contains the particle traces which survived the procedure. Check them out using ```dh.content()```.

You can look at the stream chunks that were simulated (only contains trigger channels) $\downarrow$

In [ ]:
vai.Preview(dh.get_event_iterator("trig-eff-sim").with_processing(vai.RemoveBaseline()))

You can look at the events that survived (contains trigger and passive channels) $\downarrow$

In [ ]:
vai.Preview(dh.get_event_iterator("events-eff-sim").with_processing(vai.RemoveBaseline()))

Let's have a look at the survival distribution.

In [ ]:
sim_ts = dh["efficiency_sim/timestamps"] #timestamps of simulated events 
simulated_phs = dh['efficiency_sim/simulated_phs', 0] # pulse height array we simulated
survived_trigger = dh['efficiency_sim/flag_survived_trigger'] # events triggered by the algorithm
survived_tp = dh['efficiency_sim/flag_survived_tp'] # events that are not shadowed by a testpulse

In [ ]:
vai.Histogram(
    {
        "simulated": simulated_phs, 
        "triggered": simulated_phs[survived_trigger],
        "survived tp": simulated_phs[survived_tp*survived_trigger],
    }, 
    bins=np.linspace(0, 1, 10),
    xlabel="Simulated pulse height (V)"
)

Now to get the efficiency, we need to divide the reconstructed energy by the injected energy. To recover energies from pulse heights, we need to convert with the ```e_cal``` function again. Then we binned the data and divide the histograms to get our final efficiency data.

In [ ]:
injected_voltage = simulated_phs
reconstructed_voltage = simulated_phs[survived_tp*survived_trigger]

injected_energies = my_ecal.inverse(sim_ts, injected_voltage)
reconstructed_energies = my_ecal.inverse(sim_ts, reconstructed_voltage)

In [ ]:
bins = np.linspace(0,1,200)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

In [ ]:
injected_energies_binned, bin_edges = np.histogram(injected_energies, bins=bins)

In [ ]:
reconstructed_energies_binned, bin_edges = np.histogram(reconstructed_energies, bins=bins)

In [ ]:
efficiency = reconstructed_energies_binned / np.where(injected_energies_binned !=0, injected_energies_binned, np.nan)

In [ ]:
vai.Scatter(x=bin_centers, 
            y=efficiency,
            xlabel='Pulse heights (V)',
            ylabel='Trigger efficiency',
            )

### Fitting the data with the analytical expression of the efficiency

In [ ]:
def efficiency_fct(x, p1, p2, E_thr, sigma_thr):
    return (1-p1)/2. * (1 + sp.special.erf((x - E_thr)/np.sqrt(2)/sigma_thr)) + p2

In [ ]:
popt, pcov = sp.optimize.curve_fit(efficiency_fct, 
                                   bin_centers, 
                                   efficiency,
                                   method='trf')

In [ ]:
trigger_eff_fit = efficiency_fct(bin_centers, popt[0], popt[1], popt[2], popt[3])

In [ ]:
vai.Line(x=bin_centers, 
            y=trigger_eff_fit,
            xlabel='Pulse heights (V)',
            ylabel='Trigger efficiency',
            )

## Cut efficiency

... explain how one would now apply all the quality cuts to the survived events
... give a code example of how people can plot a stacked efficiency curve for trigger and cut efficiencies

The cut efficiency is similar to the trigger effienciency. But in this case, you want to evaluate **how much particle events are removed by your quality cuts**. To do so, same method. We will use the group of simulated events ```efficiency_sim``` that we just created for the trigger efficiency and we will apply all the quality cuts we perform during the analysis.

To treat the simulated events as real events, we first need to compute the main parameters.

In [ ]:
dh.cmp("efficiency_sim")

Then, we just apply all cuts we applied during our analysis. 

In [ ]:
#TODO apply quality cuts 

decay_cuts = dh["events-eff-sim/decay_time",0]<100
delta_spike_cut = (dh["events-eff-sim/min_derivative",0]/dh["events-eff-sim/var",0])>-100
min_deriv = (dh["events-eff-sim/min_derivative",0]/dh["events-eff-sim/var",0])<-4

quality_cuts = decay_cuts*delta_spike_cut*min_deriv

If needed, we can also run the parametric fit.

In [ ]:
#TODO apply parametric fit

Then we have to follow the same steps as for the trigger efficiency:
- We take the reconstructed pulse heights and apply them all the quality cuts from our analysis.
- Then, we convert the voltages to energies with the ```e_cal``` function.
- Finally, we divide the distribution of what's comes out over what's comes in to get the efficiencies. 

```{tip}
Instead of applying all quality cuts at once, you can also add one after the other to visualise the effect of each on the final cut efficiencies. 
```

In [ ]:
reconstructed_voltage = simulated_phs[survived_tp*survived_trigger]

In [ ]:
reconstructed_voltage_after_cuts = reconstructed_voltage[quality_cuts]

In [ ]:
bins = np.linspace(0,1,200)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

In [ ]:
# Converting to energies 
reco_energies = my_ecal.inverse(sim_ts, reconstructed_voltage)
reco_energies_after_cuts = my_ecal.inverse(sim_ts, reconstructed_voltage_after_cuts)

In [ ]:
# Binning the data
reco_energies_binned = np.histogram(reco_energies, bins=bins)
reco_energies_after_cuts_binned = np.histogram(reco_energies_after_cuts, bins=bins)

In [ ]:
cut_efficiency = reco_energies_after_cuts_binned / np.where(reco_energies_binned !=0, reco_energies_binned, np.nan)

### Fitting the data with the analytical expression of the efficiency

In [ ]:
popt, pcov = sp.optimize.curve_fit(efficiency_fct, 
                                   bin_centers, 
                                   cut_efficiency,
                                   method='trf')

In [ ]:
cut_eff_fit = efficiency_fct(bin_centers, popt[0], popt[1], popt[2], popt[3])

In [ ]:
vai.Line(x=bin_centers, 
            y=cut_eff_fit,
            xlabel='Pulse heights (V)',
            ylabel='Cut efficiency',
            )

## Tips and common mistakes

... if you can think of any tips/tricks that may be useful, we can collect them here. Also mention common mistakes/errors/pitfalls.

## Other channel configurations

... maybe here we could discuss how the situation would change if we have multiple trigger channels or when using a 2d-of, e.g.
... it is probably fine to put the discussion on how to simulate SEV shifts here (and not clutter the discussion above)